In [1]:
import xarray as xr
import numpy as np

xr.set_options(keep_attrs=True)

In [43]:
# This file was made by opening the 2019 I06 bottle file in ODV, and immediately exporting it again
with xr.open_dataset("325020190403_hy1_odv.nc") as f:
    odv_data = f.load()

Things I can think of that need to change from the ODV output:
* rename dims to match argo
* Fix "strings" for the string things
* Fix flag values to be the bytes for 1-9 rather than the ASCII codes for "1"-"9"
* Add standard names and units
* remove empty/unused attrs (comments, units, etc)
* Add all the other attributes we want to use

In [44]:
# renaming dims is super easy
renamed_dims = odv_data.rename({"N_SAMPLES": "N_LEVELS", "N_STATIONS": "N_PROF"})

In [45]:
# this fixes the strings "in place" but doesn't actually do anything untill the file is saved
for name, var in renamed_dims.items():
    dtype = var.encoding.get("dtype")
    if np.dtype(dtype).type == np.string_:
        var.encoding["dtype"] = str

In [86]:
# shift the flags to "actual ints"
# requires that "keep_attrs" is True (it really should be)
modified_flags = {}
for name, var in renamed_dims.filter_by_attrs(standard_name=lambda x: x == "status_flag").items():
    modified_flags[name] = var - 48
    modified_flags[name].attrs["flag_values"] = modified_flags[name].attrs["flag_values"] - 48
    modified_flags[name].encoding = var.encoding
    modified_flags[name].encoding["_FillValue"] = 9
modified_flags = renamed_dims.merge(modified_flags, overwrite_vars=modified_flags.keys())

In [97]:
# move long_name to whp_name and units to whp_units
# remove empty units, ignore variables with standard name already
modified_std_names = {}
for name, var in modified_flags.filter_by_attrs(standard_name=lambda x: x is None).items():
    modified_std_names[name] = var.copy()
    if var.attrs["long_name"] in whp_to_stdname:
        modified_std_names[name].attrs["whp_parameter"] = modified_std_names[name].attrs["long_name"]
        modified_std_names[name].attrs["whp_units"] = modified_std_names[name].attrs["units"]
        modified_std_names[name].attrs["standard_name"] = whp_to_stdname[var.attrs["long_name"]]
        modified_std_names[name].attrs["units"] = whp_to_udunits[var.attrs["long_name"]]
        del modified_std_names[name].attrs["long_name"]
modified_params_units = modified_flags.merge(modified_std_names, overwrite_vars=modified_std_names.keys())
modified_params_units.to_netcdf('cchdo_option1.nc')

In [79]:
whp_to_stdname = {
"CTDPRS": "sea_water_pressure",
"CTDTMP": "sea_water_temperature",
"CTDSAL": "sea_water_practical_salinity",
"SALNTY": "sea_water_practical_salinity",
"OXYGEN": "moles_of_oxygen_per_unit_mass_in_sea_water",
"PHSPHT": "moles_of_phosphate_per_unit_mass_in_sea_water",
"SILCAT": "moles_of_silicate_per_unit_mass_in_sea_water",
"NITRAT": "moles_of_nitrate_per_unit_mass_in_sea_water",
"NITRIT": "moles_of_nitrite_per_unit_mass_in_sea_water",
"CFC-11": "moles_of_cfc11_per_unit_mass_in_sea_water",
"CFC-12": "moles_of_cfc12_per_unit_mass_in_sea_water", # not in CF List
"SF6": "moles_of_sf6_per_unit_mass_in_sea_water", # not in CF list
"ALKALI": "sea_water_alkalinity_expressed_as_molal_equivalent", # not in CF
"TCARBN": "moles_of_dissolved_inorganic_carbon_per_unit_mass_in_sea_water", # not in CF
"PH_TOT": "sea_water_ph_reported_on_total_scale",
"DOC": "moles_of_dissolved_organic_carbon_per_unit_mass_in_sea_water", # not in CF
"TOC": "moles_of_organic_carbon_per_unit_mass_in_sea_water", # not in CF
"DON": "moles_of_organic_nitrogen_per_unit_mass_in_sea_water", # not in CF
"TDN": "moles_of_dissolved_organic_nitrogen_per_unit_mass_in_sea_water",
"DELC14": "enrichment_of_14C_in_carbon_dioxide_in_sea_water_expressed_as_uppercase_delta_14C",
"DELC13": "enrichment_of_13C_in_carbon_dioxide_in_sea_water_expressed_as_uppercase_delta_13C", #not in CF
"DELO18": "enrichment_of_18O_in_water_in_sea_water_expressed_as_uppercase_delta_18O", # not in CF (guess)
"CTDOXY": "moles_of_oxygen_per_unit_mass_in_sea_water",
"PH_TMP": "sea_water_analysis_temperature", #not in CF (need to ask)
"N2O": "moles_of_n2o_per_unit_mass_in_sea_water", #not in CF
"REFTMP": "sea_water_temperature",
"CTDBACKSCATTER": "volume_backwards_scattering_coefficient_of_radiative_flux_in_sea_water", #need to check final units... but volts
"CTDFLUOR": "mass_concentration_of_chlorophyll_a_in_sea_water",
"CTDXMISS": "volume_beam_attenuation_coefficient_of_radiative_flux_in_sea_water_corrected_for_pure_water_attenuance",
"BTL_LON": "longitude",
"BTL_LAT": "latitude",
}
whp_to_udunits = {
"CTDPRS": "dbar",
"CTDTMP": "degc",
"CTDSAL": "1",
"SALNTY": "1",
"OXYGEN": "umol kg-1",
"PHSPHT": "umol kg-1",
"SILCAT": "umol kg-1",
"NITRAT": "umol kg-1",
"NITRIT": "umol kg-1",
"CFC-11": "pmol kg-1",
"CFC-12": "pmol kg-1", # not in CF List
"SF6": "fmol kg-1", # not in CF list
"ALKALI": "umol kg-1", # not in CF
"TCARBN": "umol kg-1", # not in CF
"PH_TOT": "1",
"DOC": "umol kg-1", # not in CF
"TOC": "umol kg-1", # not in CF
"DON": "umol kg-1", # not in CF
"TDN": "umol kg-1",
"DELC14": "0.001",
"DELC13": "0.001", #not in CF
"DELO18": "0.001", # not in CF (guess)
"CTDOXY": "umol kg-1",
"PH_TMP": "degc", #not in CF (need to ask)
"N2O": "nmol kg-1", #not in CF
"REFTMP": "degc",
"CTDBACKSCATTER": "volts", #need to check final units... but volts
"CTDFLUOR": "volts",
"CTDXMISS": "volts",
"BTL_LON": "degrees_east",
"BTL_LAT": "degrees_north",
}

In [98]:
!ncdump -h cchdo_option1.nc

netcdf cchdo_option1 {
dimensions:
	N_PROF = 55 ;
	N_LEVELS = 36 ;
variables:
	string metavar1(N_PROF) ;
		metavar1:long_name = "Cruise" ;
		metavar1:units = "" ;
		metavar1:comment = "" ;
	string metavar2(N_PROF) ;
		metavar2:long_name = "Station" ;
		metavar2:units = "" ;
		metavar2:comment = "" ;
	string metavar3(N_PROF) ;
		metavar3:long_name = "Type" ;
		metavar3:units = "" ;
		metavar3:comment = "" ;
	float longitude(N_PROF) ;
		longitude:_FillValue = -1.e+10f ;
		longitude:long_name = "Longitude" ;
		longitude:standard_name = "longitude" ;
		longitude:units = "degrees_east" ;
		longitude:comment = "" ;
		longitude:C_format = "%.3f" ;
		longitude:FORTRAN_format = "F12.3" ;
	float latitude(N_PROF) ;
		latitude:_FillValue = -1.e+10f ;
		latitude:long_name = "Latitude" ;
		latitude:standard_name = "latitude" ;
		latitude:units = "degrees_north" ;
		latitude:comment = "" ;
		latitude:C_format = "%.3f" ;
		latitude:FORTRAN_format = "F12.3" ;
	double date_time(N_PROF) ;
		date_time:_Fi

In [99]:
!ncdump -h example_cchdo_utf8.nc

netcdf example_cchdo_utf8 {
dimensions:
	N_PROF = 3 ;
	N_LEVEL = 36 ;
	N_WAVELENGTH0 = 3 ;
variables:
	float pressure(N_PROF, N_LEVEL) ;
		pressure:_FillValue = NaNf ;
		pressure:standard_name = "sea_water_pressure" ;
		pressure:units = "dbar" ;
		pressure:axis = "Z" ;
		pressure:positive = "down" ;
		pressure:whp_name = "CTDPRS" ;
		pressure:whp_unit = "DBAR" ;
	float latitude(N_PROF) ;
		latitude:standard_name = "latitude" ;
		latitude:units = "degrees_north" ;
		latitude:axis = "Y" ;
	float longitude(N_PROF) ;
		longitude:standard_name = "latitude" ;
		longitude:units = "degrees_east" ;
		longitude:axis = "X" ;
	int64 date(N_PROF) ;
		date:standard_name = "time" ;
		date:axis = "T" ;
		date:units = "minutes since 1970-01-01T00:00:00+00:00" ;
		date:calendar = "proleptic_gregorian" ;
	int wavelength0(N_WAVELENGTH0) ;
		wavelength0:standard_name = "radiation_wavelength" ;
		wavelength0:units = "nm" ;
	float var0(N_PROF, N_LEVEL) ;
		var0:_FillValue = NaNf ;
		var0:ancillary_variables 